# 04 — Grid Search of Dynamics

Holding geometry fixed (5d torus, uniform), vary the three forming-force
coefficients on a 3×3×3 grid, 3 replicates each = 81 runs × 500 steps.
Records constraint + 3 components every 100 steps.

In [ ]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from abm_core import init_torus_uniform
from experiment_grid_search import run_grid_cell

# Fixed factors
N = 300
D = 5
BUDGET = 10
N_STEPS = 500
SNAPSHOT_TIMES = [0, 100, 200, 300, 400, 500]
REPLICATES = 3

# Varied factors
LEVELS = {
    "b_homophily":  [0.0, 1.0, 3.0],
    "b_triadic":    [0.0, 0.5, 1.5],
    "b_popularity": [0.0, 0.2, 0.6],
}

OUT_DIR = Path("simulations/grid_search_torus_5d")
RUNS_DIR = OUT_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = OUT_DIR / "summary.parquet"


In [ ]:
# One init per replicate (block design): all cells share an init within a block,
# different blocks use different inits.
inits = [init_torus_uniform(n=N, d=D, rng=np.random.default_rng(seed))
         for seed in range(REPLICATES)]

cells = list(product(LEVELS["b_homophily"],
                     LEVELS["b_triadic"],
                     LEVELS["b_popularity"]))
print(f"{len(cells)} cells × {REPLICATES} replicates = {len(cells) * REPLICATES} runs")


In [ ]:
all_rows = []
total = len(cells) * REPLICATES
with tqdm(total=total, desc="grid runs") as pbar:
    for cell_idx, (b_h, b_t, b_p) in enumerate(cells):
        for rep in range(REPLICATES):
            sim_seed = rep * 1000 + cell_idx
            out_path = RUNS_DIR / f"cell_{cell_idx:02d}_rep{rep}.npz"
            rows = run_grid_cell(
                init_result=inits[rep],
                b_homophily=b_h,
                b_triadic=b_t,
                b_popularity=b_p,
                budget=BUDGET,
                n_steps=N_STEPS,
                snapshot_times=SNAPSHOT_TIMES,
                sim_seed=sim_seed,
                out_path=out_path,
            )
            for row in rows:
                row["cell_id"] = cell_idx
                row["replicate"] = rep
            all_rows.extend(rows)
            pbar.update(1)

summary = pd.DataFrame(all_rows)
summary.to_parquet(SUMMARY_PATH)
print(f"Saved {len(summary)} rows → {SUMMARY_PATH}")
summary.head()


In [ ]:
summary = pd.read_parquet(SUMMARY_PATH)

# Baseline cell: no mechanisms — almost all nodes isolated → mean_constraint ≈ 1
baseline = summary[
    (summary["b_homophily"] == 0)
    & (summary["b_triadic"] == 0)
    & (summary["b_popularity"] == 0)
    & (summary["t"] == N_STEPS)
]
print("Baseline (all-zero) at t=500, mean_constraint per replicate:")
print(baseline[["replicate", "mean_constraint", "std_constraint"]])
assert baseline["mean_constraint"].mean() > 0.9, "baseline should be near 1.0 (mostly isolated)"

# High homophily-only cell — should differ clearly
homophily_only = summary[
    (summary["b_homophily"] == 3.0)
    & (summary["b_triadic"] == 0)
    & (summary["b_popularity"] == 0)
    & (summary["t"] == N_STEPS)
]
print("\nHomophily-only at t=500:")
print(homophily_only[["replicate", "mean_constraint", "std_constraint"]])


In [ ]:
def plot_trajectories(summary, color_by, metric="mean_constraint"):
    fig, ax = plt.subplots(figsize=(8, 5))
    grouped = summary.groupby(["b_homophily", "b_triadic", "b_popularity", "t"])
    agg = grouped[metric].agg(["mean", "std"]).reset_index()
    for level, subset in agg.groupby(color_by):
        for cell_key, line in subset.groupby(
            [c for c in ("b_homophily", "b_triadic", "b_popularity") if c != color_by]
        ):
            ax.plot(line["t"], line["mean"], alpha=0.4)
        # bold mean across cells with this level for the legend
        bold = subset.groupby("t")["mean"].mean()
        ax.plot(bold.index, bold.values, label=f"{color_by}={level}", linewidth=2.5)
    ax.set_xlabel("t")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} over time, colored by {color_by}")
    ax.legend()
    plt.tight_layout()
    plt.show()

for color_by in ("b_homophily", "b_triadic", "b_popularity"):
    plot_trajectories(summary, color_by, metric="mean_constraint")


In [ ]:
def plot_histograms_at_t(snapshot_t, metric, slice_by):
    """Pool per-node values across replicates, plot histogram per level of slice_by."""
    levels = sorted(LEVELS[slice_by])
    fig, axes = plt.subplots(1, len(levels), figsize=(4 * len(levels), 4), sharey=True)
    for ax, lvl in zip(axes, levels):
        # Pool across all cells with slice_by==lvl and across replicates
        cell_indices = [i for i, c in enumerate(cells)
                        if c[("b_homophily", "b_triadic", "b_popularity").index(slice_by)] == lvl]
        snap_idx = SNAPSHOT_TIMES.index(snapshot_t)
        all_vals = []
        for cell_idx in cell_indices:
            for rep in range(REPLICATES):
                data = np.load(RUNS_DIR / f"cell_{cell_idx:02d}_rep{rep}.npz")
                all_vals.append(data[metric][snap_idx])
        pooled = np.concatenate(all_vals)
        ax.hist(pooled, bins=40, alpha=0.7)
        ax.set_title(f"{slice_by}={lvl}")
        ax.set_xlabel(metric)
    fig.suptitle(f"{metric} histogram at t={snapshot_t}, sliced by {slice_by}")
    plt.tight_layout()
    plt.show()

for metric in ("constraint", "c_size", "c_density", "c_hierarchy"):
    plot_histograms_at_t(snapshot_t=N_STEPS, metric=metric, slice_by="b_homophily")
